In [1]:
!pip install torch transformers sentence-transformers faiss-cpu


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 57.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/30.7 MB 14.1 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstallin

In [2]:
import requests
import re
import json
from tqdm import tqdm

# List of book URLs (Plain Text Versions)
BOOK_URLS = [
    "https://www.gutenberg.org/ebooks/56640.txt.utf-8",
    "https://www.gutenberg.org/ebooks/67813.txt.utf-8",
    "https://www.gutenberg.org/ebooks/20772.txt.utf-8",
    "https://www.gutenberg.org/ebooks/40190.txt.utf-8",
    "https://www.gutenberg.org/ebooks/4924.txt.utf-8",
    "https://www.gutenberg.org/ebooks/4525.txt.utf-8",
    "https://www.gutenberg.org/ebooks/40190.txt.utf-8"
]

# Step 1: Download Books
def download_books(urls):
    """Downloads books and saves them locally."""
    for i, url in enumerate(tqdm(urls, desc="Downloading")):
        response = requests.get(url)
        if response.status_code == 200:
            with open(f"book_{i+1}.txt", "w", encoding="utf-8") as f:
                f.write(response.text)

def clean_text(text):
    """Removes headers, footers, disclaimers, and unwanted content from the text."""
    patterns = [
        r"(?s)^.?START OF (THE|THIS) PROJECT GUTENBERG EBOOK.?\n",
        r"(?s)END OF (THE|THIS) PROJECT GUTENBERG EBOOK.*$",
        r"(?s)This ebook is for the use of anyone anywhere.*?restrictions whatsoever\." ,
        r"(?s)Produced by.?www.pgdp.net.?\n",
        r"(?s)Transcriber’s Note:.*?\n",
        r"(?s)Online Distributed Proofreading Team.*?\n",
        r"(?s)Transcriber’s Notes.*?\n",
        r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b"
    ]
    for pattern in patterns:
        text = re.sub(pattern, "", text)
    return text.replace("\ufeff", "").replace("\r", "").strip()

# Step 2: Process Books
def process_books(urls):
    """Reads, cleans, and processes downloaded books."""
    processed_books = []
    for i in range(len(urls)):
        with open(f"book_{i+1}.txt", "r", encoding="utf-8") as f:
            cleaned_text = clean_text(f.read())
            processed_books.append(cleaned_text)
    return processed_books

def split_into_paragraphs(text, min_length=300):
    """Splits text into paragraphs and removes short ones."""
    return [p.strip() for p in re.split(r"\n{2,}", text) if len(p) > min_length]

def filter_unwanted_paragraphs(paragraphs):
    """Filters out unwanted paragraphs like disclaimers and legal text."""
    unwanted_starts = {
        "the project gutenberg", "this ebook is for the use of", "release date:",
        "language:", "credits:", "produced by", "transcriber’s note:",
        "text printed in", "if you are not located in the united states"
    }
    return [p for p in paragraphs if not any(phrase in p.lower().split()[:5] for phrase in unwanted_starts)]

# Step 3: Extract and Filter Paragraphs
def extract_paragraphs(processed_books):
    """Extracts and filters paragraphs from processed books."""
    all_paragraphs = []
    for book in processed_books:
        paragraphs = split_into_paragraphs(book)
        filtered_paragraphs = filter_unwanted_paragraphs(paragraphs)
        all_paragraphs.extend(filtered_paragraphs)
    return all_paragraphs

def save_to_json(data, filename="preprocessed_books.json"):
    """Saves processed paragraphs to a JSON file."""
    with open(filename, "w", encoding="utf-8") as f:
        json.dump({"paragraphs": data}, f, indent=4)
    print("Preprocessed data saved in preprocessed_books.json!")

# Main Execution
def main():
    download_books(BOOK_URLS)
    processed_books = process_books(BOOK_URLS)
    all_paragraphs = extract_paragraphs(processed_books)
    save_to_json(all_paragraphs)

if __name__ == "__main__":
    main()


Downloading: 100%|██████████| 7/7 [00:05<00:00,  1.23it/s]


Preprocessed data saved in preprocessed_books.json!


In [3]:
import json
import torch
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

# Load preprocessed paragraphs
with open("preprocessed_books.json", "r", encoding="utf-8") as file:
    data = json.load(file)
paragraphs = data["paragraphs"]

# Initialize Sentence Transformer model
model_name = "sentence-transformers/multi-qa-mpnet-base-dot-v1"
device = "cuda" if torch.cuda.is_available() else "cpu"
embedding_model = SentenceTransformer(model_name, device=device)

# Define file paths
embedding_path = "paragraph_embeddings.npy"
faiss_index_path = "faiss_index.bin"

# Attempt to load precomputed embeddings and FAISS index
try:
    paragraph_embeddings_np = np.load(embedding_path)
    index = faiss.read_index(faiss_index_path)
    print(" Precomputed embeddings and FAISS index successfully loaded!")
except FileNotFoundError:
    print(" No precomputed data found. Generating embeddings...")

    # Compute embeddings in manageable batches
    batch_size = 16  # Adjust for available memory
    paragraph_embeddings = embedding_model.encode(
        paragraphs, convert_to_numpy=True, batch_size=batch_size, show_progress_bar=True
    )

    # Save computed embeddings
    paragraph_embeddings_np = np.array(paragraph_embeddings)
    np.save(embedding_path, paragraph_embeddings_np)

    # Construct and save FAISS index
    index = faiss.IndexFlatIP(paragraph_embeddings_np.shape[1])
    index.add(paragraph_embeddings_np)
    faiss.write_index(index, faiss_index_path)
    print(" Embeddings computed and FAISS index saved.")

# Function to retrieve top-k matching paragraphs
def retrieve_top_paragraphs(query, top_k=3):
    query_embedding = embedding_model.encode(query, convert_to_numpy=True).reshape(1, -1)
    scores, indices = index.search(query_embedding, top_k)
    return [(paragraphs[i], scores[0][idx]) for idx, i in enumerate(indices[0])]

# Example query
query_text = "What is the importance of cultivation?"
results = retrieve_top_paragraphs(query_text)

# Display retrieved results
print("\n Most Relevant Paragraphs:")
for idx, (paragraph, similarity) in enumerate(results, start=1):
    print(f"{idx}. {paragraph[:200]}... (Similarity: {similarity:.4f})")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/212 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/8.71k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

 No precomputed data found. Generating embeddings...


Batches:   0%|          | 0/270 [00:00<?, ?it/s]

 Embeddings computed and FAISS index saved.

 Most Relevant Paragraphs:
1. Second, the gardener must cultivate his rich land most carefully and
economically. He crowds his land with products that must grow apace.
Therefore he, least of all growers, can afford to have any of ... (Similarity: 24.5165)
2. In the earlier pages of this book you were told something about the food
of plants. One of the main elements of plant food, perhaps you remember,
is nitrogen. Just as soon as the roots of the legumino... (Similarity: 23.4830)
3. Irrigated lands should be carefully and thoroughly tilled. The water for
irrigation is costly, and should be made to go as far as possible. Good
tillage saves the water. Moreover, all cultivated crops... (Similarity: 23.4262)


In [4]:
!pip install torch transformers


In [5]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# Load embedding model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Assume 'index' is a pre-built FAISS index and 'paragraphs' is a list of texts
def retrieve_relevant_paragraphs(query, top_k=2):
    query_embedding = embedding_model.encode([query])
    _, indices = index.search(np.array(query_embedding, dtype=np.float32), top_k)
    return [paragraphs[i] for i in indices[0]]


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [6]:
import torch
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
from transformers import (
    BertTokenizer, BertForQuestionAnswering,
    BartTokenizer, BartForConditionalGeneration
)


# Select Device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Load Embedding Model for Retrieval
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Load BERT Question-Answering Model
print("Loading BERT-QA model...")
qa_model = BertForQuestionAnswering.from_pretrained(
    "bert-large-uncased-whole-word-masking-finetuned-squad"
).to(device)
qa_tokenizer = BertTokenizer.from_pretrained("bert-large-uncased-whole-word-masking-finetuned-squad")

# Load BART Summarization Model
print("Loading BART model for summarization...")
summarization_model = BartForConditionalGeneration.from_pretrained("facebook/bart-base").to(device)
summarization_model.half()  # Use FP16 for efficiency
summarization_tokenizer = BartTokenizer.from_pretrained("facebook/bart-base")


# Sample list of paragraphs (Replace with actual dataset)
paragraphs = [
    "Dry-farming is a set of agricultural techniques for non-irrigated cultivation...",
    "Soil conservation is important for maintaining agricultural productivity...",
    "Crop rotation helps in soil fertility management by alternating different crops...",
]

# Encode all paragraphs into embeddings
print("Generating embeddings for FAISS index...")
paragraph_embeddings = np.array(embedding_model.encode(paragraphs), dtype=np.float32)

# Create and populate FAISS index
index = faiss.IndexFlatL2(paragraph_embeddings.shape[1])  # L2 distance (Euclidean)
index.add(paragraph_embeddings)
print("FAISS index built successfully!")

def retrieve_relevant_paragraphs(query, top_k=2):
    """Retrieves top_k most relevant paragraphs from FAISS index."""
    print(f"\nSearching for relevant content related to: '{query}'...")
    query_embedding = np.array(embedding_model.encode([query]), dtype=np.float32)
    _, indices = index.search(query_embedding, top_k)  # Retrieve top_k results
    return [paragraphs[i] for i in indices[0]]  # Return matching paragraphs

def answer_question(question, context):
    """Extracts an answer from the given context based on the question."""
    print(f"\nAnswering question: '{question}'...")

    inputs = qa_tokenizer(question, context, return_tensors="pt", max_length=512, truncation=True).to(device)

    with torch.no_grad():
        outputs = qa_model(**inputs)

    answer_start = torch.argmax(outputs.start_logits)
    answer_end = torch.argmax(outputs.end_logits) + 1

    answer = qa_tokenizer.convert_tokens_to_string(
        qa_tokenizer.convert_ids_to_tokens(inputs["input_ids"][0][answer_start:answer_end])
    )

    return answer.strip()

def summarize_text(text, max_length=100):
    """Generates a concise summary of the input text."""
    print("\nSummarizing text...")
    input_text = "summarize: " + text
    input_ids = summarization_tokenizer.encode(
        input_text, return_tensors="pt", max_length=512, truncation=True
    ).to(device)

    summary_ids = summarization_model.generate(
        input_ids, max_length=max_length, min_length=30, do_sample=False, num_beams=1
    )

    summary = summarization_tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    print("Summary successfully generated!")
    return summary

# User query
query = "cultivation"
retrieved_paragraphs = retrieve_relevant_paragraphs(query, top_k=1)

if retrieved_paragraphs:
    context = retrieved_paragraphs[0]  # Extract paragraph text
    question = "What is the importance of cultivation?"

    print("\nExtracted Context (First 300 characters):")
    print(f"{context[:300]}...\n")

    # Get Answer
    answer = answer_question(question, context)

    # Summarize Context
    summary = summarize_text(context)

    print("\n🔹 Question:", question)
    print("🔹 Extracted Answer:", answer if answer else "No clear answer found in the text.")
    print("\n🔹 Summary of Retrieved Text:\n", summary)
else:
    print("No relevant paragraphs found for the given query.")


Using device: cuda
Loading BERT-QA model...


config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.34G [00:00<?, ?B/s]

Some weights of the model checkpoint at bert-large-uncased-whole-word-masking-finetuned-squad were not used when initializing BertForQuestionAnswering: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForQuestionAnswering from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForQuestionAnswering from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Generating embeddings for FAISS index...
FAISS index built successfully!

Searching for relevant content related to: 'cultivation'...

Extracted Context (First 300 characters):
Dry-farming is a set of agricultural techniques for non-irrigated cultivation......


Answering question: 'What is the importance of cultivation?'...

Summarizing text...


/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:676: UserWarning: `num_beams` is set to 1. However, `early_stopping` is set to `True` -- this flag is only used in beam-based generation modes. You should set `num_beams>1` or unset `early_stopping`.
  warnings.warn(


Summary successfully generated!

🔹 Question: What is the importance of cultivation?
🔹 Extracted Answer: non - irrigated

🔹 Summary of Retrieved Text:
 summarize: Dry-farming is a set of agricultural techniques for non-irrigated cultivation...... and the use of dry-farms...


In [7]:
!pip install gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 MB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.1/322.1 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.9/94.9 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 115.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 5.9 MB/s eta 0:00:00
  Attempting uninstall: markupsafe
    Found existing installation: MarkupSafe 3.0.2
    Uninstalling MarkupSafe-3.0.2:
      Successfully uninstalled MarkupSafe-3.0.2


In [8]:
!pip install datasets


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.4/485.4 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 14.6 MB/s eta 0:00:00


In [9]:
import json
import torch
from datasets import Dataset
from transformers import BartTokenizer, BartForConditionalGeneration, Trainer, TrainingArguments

# Load preprocessed paragraphs
print(" Loading dataset...")
with open("preprocessed_books.json", "r", encoding="utf-8") as f:
    data = json.load(f)

paragraphs = data.get("paragraphs", [])
if not paragraphs:
    raise ValueError(" No paragraphs found in the dataset!")

print(f" Successfully loaded {len(paragraphs)} paragraphs.")

# Generate artificial summaries (or use real ones if available)
summaries = [" Summary: " + p[:100] + "..." for p in paragraphs]

# Create dataset
dataset = Dataset.from_dict({"document": paragraphs, "summary": summaries})

# Split dataset (80% training, 20% validation)
dataset = dataset.train_test_split(test_size=0.2)
print(f" Dataset split: {len(dataset['train'])} training samples, {len(dataset['test'])} validation samples.")

# Load tokenizer
print(" Loading tokenizer...")
tokenizer = BartTokenizer.from_pretrained("facebook/bart-base")
print(" Tokenizer ready.")

# Tokenization function
def tokenize_function(examples):
    inputs = tokenizer(examples["document"], max_length=512, truncation=True, padding="max_length")
    targets = tokenizer(examples["summary"], max_length=128, truncation=True, padding="max_length")
    inputs["labels"] = targets["input_ids"]
    return inputs

# Apply tokenization
print(" Tokenizing dataset...")
tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=["document", "summary"])
print(" Tokenization completed.")

# Load model
print(" Loading BART model for fine-tuning...")
model = BartForConditionalGeneration.from_pretrained("facebook/bart-base")
print(" Model successfully loaded.")

# Define training arguments
training_args = TrainingArguments(
    output_dir="./bart-finetuned",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,  # Adjust based on available memory
    per_device_eval_batch_size=4,
    num_train_epochs=3,  # Modify as needed
    weight_decay=0.01,
    save_total_limit=2,
    push_to_hub=False,
    report_to="none",  # Disables logging to external services
)

# Trainer setup
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
)

# Start fine-tuning
print(" Fine-tuning process started...")
trainer.train()
print(" Fine-tuning completed successfully!")

# Save model & tokenizer
print(" Saving fine-tuned model and tokenizer...")
model.save_pretrained("bart-finetuned-agriculture")
tokenizer.save_pretrained("bart-finetuned-agriculture")
print(" Model and tokenizer saved successfully!")

 Loading dataset...
 Successfully loaded 4306 paragraphs.
 Dataset split: 3444 training samples, 862 validation samples.
 Loading tokenizer...
 Tokenizer ready.
 Tokenizing dataset...


Map:   0%|          | 0/3444 [00:00<?, ? examples/s]

Map:   0%|          | 0/862 [00:00<?, ? examples/s]

 Tokenization completed.
 Loading BART model for fine-tuning...
 Model successfully loaded.


/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


 Fine-tuning process started...


Epoch,Training Loss,Validation Loss
1,1.155200,0.048865
2,0.045400,0.042143
3,0.034400,0.042367


/usr/local/lib/python3.11/dist-packages/transformers/modeling_utils.py:2758: UserWarning: Moving the following attributes in the config to the generation config: {'early_stopping': True, 'num_beams': 4, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


 Fine-tuning completed successfully!
 Saving fine-tuned model and tokenizer...
 Model and tokenizer saved successfully!


In [10]:
!pip install datasets
import json
import torch
from datasets import Dataset
from transformers import BertTokenizerFast, BertForQuestionAnswering, Trainer, TrainingArguments

#Step 1: Load Preprocessed Data
print(" Loading preprocessed book paragraphs...")
with open("preprocessed_books.json", "r", encoding="utf-8") as f:
    data = json.load(f)

paragraphs = data.get("paragraphs", [])

if not paragraphs:
    raise ValueError("No paragraphs found in preprocessed_books.json!")

#Step 2: Create Synthetic QA Pairs (Replace with Real Annotations If Available)
print("Creating synthetic QA dataset...")
qa_data = [
    {
        "context": p,
        "question": "What is this passage about?",
        "answer": p[:50],  # Dummy answer (first 50 chars)
        "answer_start": 0
    }
    for p in paragraphs[:500]  # Limit to first 500 paragraphs for training
]

#Step 3: Convert to Hugging Face Dataset
print("Converting data into Hugging Face Dataset format...")
train_data = {
    "context": [d["context"] for d in qa_data],
    "question": [d["question"] for d in qa_data],
    "answers": [{"text": [d["answer"]], "answer_start": [d["answer_start"]]} for d in qa_data]
}

dataset = Dataset.from_dict(train_data)

#Step 4: Split Dataset into Train & Validation (80%-20%)
dataset = dataset.train_test_split(test_size=0.2)
print(f"✅ Dataset split completed. Train size: {len(dataset['train'])}, Test size: {len(dataset['test'])}")

#Step 5: Load Tokenizer
print("🔡 Loading BERT tokenizer...")
tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")

#Step 6: Tokenization Function
def tokenize_function(examples):
    tokenized_inputs = tokenizer(
        examples["question"],
        examples["context"],
        truncation="only_second",
        max_length=512,
        padding="max_length",
        return_offsets_mapping=True,
    )

    start_positions, end_positions = [], []

    for i, (offset, answer) in enumerate(zip(tokenized_inputs["offset_mapping"], examples["answers"])):
        start_char, end_char = answer["answer_start"][0], answer["answer_start"][0] + len(answer["text"][0])
        sequence_ids = tokenized_inputs.sequence_ids(i)

        context_start, context_end = sequence_ids.index(1), len(sequence_ids) - 1 - sequence_ids[::-1].index(1)

        start_pos = end_pos = 0
        for j in range(context_start, context_end):
            if offset[j][0] <= start_char and offset[j][1] >= start_char:
                start_pos = j
            if offset[j][0] <= end_char and offset[j][1] >= end_char:
                end_pos = j
                break

        start_positions.append(start_pos)
        end_positions.append(end_pos)

    tokenized_inputs["start_positions"] = start_positions
    tokenized_inputs["end_positions"] = end_positions
    tokenized_inputs.pop("offset_mapping")

    return tokenized_inputs

#Step 7: Apply Tokenization
print("Tokenizing dataset...")
tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=["context", "question", "answers"])

#Step 8: Load Pretrained BERT Model
print("Loading BERT model for Question Answering...")
model = BertForQuestionAnswering.from_pretrained("bert-base-uncased")

#Step 9: Define Training Arguments
training_args = TrainingArguments(
    output_dir="./bert-qa-finetuned",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    weight_decay=0.01,
    save_total_limit=2,
    push_to_hub=False,
    report_to="none"
)

#Step 10: Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
)

#Step 11: Start Fine-Tuning
print("Starting fine-tuning...")
trainer.train()

#Step 12: Save Fine-Tuned Model & Tokenizer
print("Saving fine-tuned model...")
model.save_pretrained("bert-qa-finetuned-agriculture")
tokenizer.save_pretrained("bert-qa-finetuned-agriculture")

print("Fine-tuning completed! Model saved to 'bert-qa-finetuned-agriculture'.")

 Loading preprocessed book paragraphs...
Creating synthetic QA dataset...
Converting data into Hugging Face Dataset format...
✅ Dataset split completed. Train size: 400, Test size: 100
🔡 Loading BERT tokenizer...


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForQuestionAnswering were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Starting fine-tuning...


Epoch,Training Loss,Validation Loss
1,No log,1.054097
2,No log,1.085628
3,No log,1.092043


Saving fine-tuned model...
Fine-tuning completed! Model saved to 'bert-qa-finetuned-agriculture'.


In [13]:
import torch
import gradio as gr
from datetime import datetime
from sentence_transformers import SentenceTransformer
from transformers import BartForConditionalGeneration, BartTokenizer, BertForQuestionAnswering, BertTokenizer

# Load preprocessed agricultural data
with open("preprocessed_books.json", "r", encoding="utf-8") as f:
    data = json.load(f)

paragraphs = data["paragraphs"]

# Load FAISS index for retrieval
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
embeddings = embedding_model.encode(paragraphs, convert_to_numpy=True)
index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)

def retrieve_relevant_paragraphs(query, top_k=1):
    query_embedding = embedding_model.encode([query])
    _, retrieved_indices = index.search(query_embedding, top_k)
    return [paragraphs[i] for i in retrieved_indices[0]]

# Load fine-tuned models
device = "cuda" if torch.cuda.is_available() else "cpu"

summarization_model = BartForConditionalGeneration.from_pretrained("bart-finetuned-agriculture").to(device)
summarization_tokenizer = BartTokenizer.from_pretrained("bart-finetuned-agriculture")

qa_model = BertForQuestionAnswering.from_pretrained("bert-qa-finetuned-agriculture").to(device)
qa_tokenizer = BertTokenizer.from_pretrained("bert-qa-finetuned-agriculture")

def summarize_text(text, max_length=100):
    input_text = "summarize: " + text
    input_ids = summarization_tokenizer.encode(input_text, return_tensors="pt", max_length=512, truncation=True).to(device)
    summary_ids = summarization_model.generate(input_ids, max_length=max_length, min_length=30, do_sample=False, num_beams=1)
    return summarization_tokenizer.decode(summary_ids[0], skip_special_tokens=True)

def answer_question(question, context):
    inputs = qa_tokenizer(question, context, return_tensors="pt", max_length=384, truncation=True).to(device)
    outputs = qa_model(**inputs)
    answer_start = torch.argmax(outputs.start_logits)
    answer_end = torch.argmax(outputs.end_logits) + 1
    answer = qa_tokenizer.convert_tokens_to_string(qa_tokenizer.convert_ids_to_tokens(inputs["input_ids"][0][answer_start:answer_end]))
    return answer

def chatbot(user_input):
    user_input = user_input.strip().lower()
    summarization_keywords = ["summarize", "tell me about", "describe"]
    question_keywords = ["what", "how", "explain", "why", "when", "where", "who"]

    if any(user_input.startswith(keyword) for keyword in summarization_keywords):
        topic = " ".join(user_input.split(" ")[1:])
        retrieved_paragraphs = retrieve_relevant_paragraphs(topic, top_k=1)
        if retrieved_paragraphs:
            summary = summarize_text(retrieved_paragraphs[0])
            return f"* Summary:* {summary}\n\n Generated on {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"
        return " Info cannot be found."

    elif any(user_input.startswith(keyword) for keyword in question_keywords):
        retrieved_paragraphs = retrieve_relevant_paragraphs(user_input, top_k=1)
        if retrieved_paragraphs:
            answer = answer_question(user_input, retrieved_paragraphs[0])
            return f"*Answer:* {answer}\n\n Context: {retrieved_paragraphs[0][:250]}...\n\n Generated on {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"
        return "Can't find answer."

    return "What is your Question?"

with gr.Blocks(theme="huggingface") as demo:
    gr.Markdown("""
    # Agriculture Chatbot
    """)

    with gr.Row():
        user_input = gr.Textbox(placeholder="Question", label="Input", interactive=True)
        send_button = gr.Button("Send")

    chatbot_output = gr.Textbox(label="Your answer", interactive=False)
    send_button.click(chatbot, inputs=user_input, outputs=chatbot_output)

demo.launch()

/usr/local/lib/python3.11/dist-packages/gradio/blocks.py:1102: UserWarning: Cannot load huggingface. Caught Exception: 404 Client Error: Not Found for url: https://huggingface.co/api/spaces/huggingface (Request ID: Root=1-67cc570b-2192e5001648f36f4cc373dc;29e9eeef-05b8-4d8b-bdef-a799d9ecd238)

Sorry, we can't find the page you are looking for.
  warnings.warn(f"Cannot load {theme}. Caught Exception: {str(e)}")


Running Gradio in a Colab notebook requires sharing enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5e49b0ce52c925d2c7.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
